In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
import os
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import datasets, transforms
import warnings

### Functions

In [4]:
# ==========================================
# CONFIG
# ==========================================
SAVE_PLOTS = True
PLOT_DIR = "./functions"
SEED = 67

if SAVE_PLOTS and not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

# Suppress overflow warnings
warnings.filterwarnings('ignore')


# ==========================================
# 1. OBJECTIVE FUNCTIONS
# ==========================================
def generate_quadratic(dim=10, cond_number=10):
    np.random.seed(SEED)
    V = np.linalg.qr(np.random.randn(dim, dim))[0]
    eigvals = np.linspace(1, cond_number, dim)
    Q = V @ np.diag(eigvals) @ V.T
    return Q

def quadratic_f(x, Q):
    return 0.5 * x.T @ Q @ x

def quadratic_grad(x, Q):
    return Q @ x

def rosenbrock_f(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

def rosenbrock_grad(x):
    dx = -2*(1 - x[0]) - 400*x[0]*(x[1] - x[0]**2)
    dy = 200*(x[1] - x[0]**2)
    return np.array([dx, dy])


# ==========================================
# 2. OPTIMIZERS
# ==========================================
def sgd_momentum(f, grad_f, x0, lr=0.001, beta=0.9, tol=1e-6, max_iter=10000):
    x = x0.copy()
    v = np.zeros_like(x)
    losses = []
    start = time.time()
    
    for _ in range(max_iter):
        loss = f(x)
        losses.append(loss)
        g = grad_f(x)
        if np.linalg.norm(g) < tol: break
        
        v = beta * v + (1 - beta) * g
        x -= lr * v
        
    return x, losses, time.time() - start

def adam(f, grad_f, x0, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8, tol=1e-6, max_iter=10000):
    x = x0.copy()
    m = np.zeros_like(x)
    v = np.zeros_like(x)
    losses = []
    start = time.time()
    
    for t in range(1, max_iter + 1):
        loss = f(x)
        losses.append(loss)
        g = grad_f(x)
        if np.linalg.norm(g) < tol: break
        
        m = beta1*m + (1-beta1)*g
        v = beta2*v + (1-beta2)*(g*g)
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        
        x -= lr * m_hat / (np.sqrt(v_hat) + eps)
        
    return x, losses, time.time() - start

def rmsprop(f, grad_f, x0, lr=0.001, beta=0.9, eps=1e-8, tol=1e-6, max_iter=10000):
    x = x0.copy()
    s = np.zeros_like(x)
    losses = []
    start = time.time()
    
    for _ in range(max_iter):
        loss = f(x)
        losses.append(loss)
        g = grad_f(x)
        if np.linalg.norm(g) < tol: break
        
        s = beta*s + (1-beta)*(g*g)
        x -= lr * g / (np.sqrt(s) + eps)
        
    return x, losses, time.time() - start


# ======================
# Adam-E 
# ======================
def dpae(f, grad_f, x0, lr=0.001, max_iter=10000, initial_explorers=6, max_explorers=12, tol=1e-6):
    eps = 1e-8
    beta1, beta2 = 0.9, 0.999
    start_time = time.time()

    def norm(v):
        n = np.linalg.norm(v)
        return v / (n + eps)

    def orthogonal(g, scale=1e-2):
        u = np.random.randn(*g.shape)
        gnorm2 = np.dot(g, g)
        if gnorm2 < 1e-12: return norm(u)*scale
        proj = (np.dot(u, g) / (gnorm2 + eps)) * g
        return norm(u - proj) * scale

    explorers = []
    for i in range(initial_explorers):
        explorers.append({
            "x": x0.copy(),
            "m": np.zeros_like(x0),
            "v": np.zeros_like(x0),
            "d0": norm(np.random.randn(*x0.shape)),
            "grad_norms": [], "losses": [],
            "improvements": [], "long_grad_sum": 0.0,
            "long_improve_sum": 0.0, "n": 0,
            "bad_streak": 0, "is_anchor": (i == 0)
        })

    best_curve = []

    for _ in range(max_iter):
        for e in explorers:
            g = grad_f(e["x"])
            loss = f(e["x"])
            e["losses"].append(loss)
            e["grad_norms"].append(np.linalg.norm(g))
            e["long_grad_sum"] += np.linalg.norm(g)
            e["n"] += 1

            e["m"] = beta1*e["m"] + (1-beta1)*g
            e["v"] = beta2*e["v"] + (1-beta2)*(g*g)
            m_hat = e["m"]/(1-beta1**e["n"])
            v_hat = e["v"]/(1-beta2**e["n"])
            
            adam_step = -lr * m_hat / (np.sqrt(v_hat) + eps)

            if e["is_anchor"]:
                explore_step = 0.0
            else:
                w = min(8, e["n"])
                Gs = np.mean(e["grad_norms"][-w:])
                Gl = e["long_grad_sum"]/e["n"] + eps
                omega = 1.0 / (1.0 + (Gs/Gl))
                p = 1.0 / (1.0 + Gl)
                explore_step = lr * omega * p * e["d0"]

            e["x"] += adam_step + explore_step

        best_loss = min(e["losses"][-1] for e in explorers)
        best_curve.append(best_loss)

    best_e = min(explorers, key=lambda e: f(e["x"]))
    return best_e["x"], best_curve, time.time() - start_time


# ==========================================
# 3. MULTI-SEED BENCHMARK ENGINE
# ==========================================

SEEDS = range(10)

# PER-FUNCTION TOLERANCES
per_func_tol = {
    "Well-Conditioned Quadratic": 1e-6,
    "Ill-Conditioned Quadratic": 1e-6,
    "Rosenbrock Function": 1e-4
}

# PER-FUNCTION ITERATIONS
per_func_iters = {
    "Well-Conditioned Quadratic": 2000,
    "Ill-Conditioned Quadratic": 20000,
    "Rosenbrock Function": 10000,
}

def safe_accuracy(fval):
    return 1.0 / (1.0 + float(fval))


def generate_plots_multi(name, stats):
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    opt_names = list(stats.keys())
    colors = ['red','green','orange','blue']

    # 1) Loss Curves mean±std
    for idx, opt in enumerate(opt_names):
        runs = stats[opt]["loss_history"]
        max_len = max(len(r) for r in runs)
        padded = np.array([
            np.pad(r, (0, max_len - len(r)), constant_values=r[-1])
            for r in runs
        ])
        mean = padded.mean(axis=0)
        std = padded.std(axis=0)

        ax[0].plot(mean, color=colors[idx], label=opt)
        ax[0].fill_between(range(max_len), mean-std, mean+std,
                           color=colors[idx], alpha=0.2)

    ax[0].set_yscale("log")
    ax[0].set_title(f"{name}: Loss (mean ± std)")
    ax[0].grid(True, which="both", alpha=0.3)
    ax[0].legend()

    # 2) Best Loss Chart
    vals = [np.mean(stats[opt]["final_losses"]) for opt in opt_names]
    stds = [np.std(stats[opt]["final_losses"]) for opt in opt_names]
    ax[1].bar(opt_names, vals, yerr=stds, color=colors, capsize=5)
    ax[1].set_yscale("log")
    ax[1].set_title(f"{name}: Best Loss (mean ± std)")
    ax[1].grid(True, axis="y", which="both", alpha=0.3)

    # 3) Time Chart
    tm = [np.mean(stats[opt]["times"]) for opt in opt_names]
    ts = [np.std(stats[opt]["times"]) for opt in opt_names]
    ax[2].bar(opt_names, tm, yerr=ts, color=colors, capsize=5)
    ax[2].set_title(f"{name}: Runtime")
    ax[2].set_ylabel("Seconds")

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, name.replace(" ","_")+"_benchmark.png"))
    plt.close()


def run_benchmark_suite_multi():
    print("\nRunning MULTI-SEED benchmark...\n")

    # --- Prepare functions ---
    Q_well = generate_quadratic(10,10)
    Q_ill  = generate_quadratic(10,1000)

    problems = [
        ("Well-Conditioned Quadratic", 
            lambda x: quadratic_f(x, Q_well),
            lambda x: quadratic_grad(x, Q_well),
            10, 0.01),

        ("Ill-Conditioned Quadratic", 
            lambda x: quadratic_f(x, Q_ill),
            lambda x: quadratic_grad(x, Q_ill),
            10, 0.001),

        ("Rosenbrock Function",
            rosenbrock_f,
            rosenbrock_grad,
            2, 0.001)
    ]

    optimizers = {
        "SGD+Mom": sgd_momentum,
        "Adam": adam,
        "RMSProp": rmsprop,
        "Adam-E": dpae
    }

    # Run each problem
    for name, f, grad, dim, lr in problems:
        print(f"\n=== {name} across {len(SEEDS)} seeds ===\n")

        stats = {
            opt: {
                "loss_history": [],
                "final_losses": [],
                "times": [],
                "iters": [],
                "acc": []
            }
            for opt in optimizers
        }

        max_iters = per_func_iters[name]
        tol_here  = per_func_tol[name]

        for seed in SEEDS:
            np.random.seed(seed)

            if "Rosenbrock" in name:
                x0 = np.array([-1.2,1.0])
            else:
                x0 = np.random.randn(dim)

            for opt_name, opt_func in optimizers.items():
                # FORCE full run: tol=0.0
                x_final, losses, duration = opt_func(
                    f, grad, x0,
                    lr=lr,
                    tol=0.0,
                    max_iter=max_iters
                )

                # compute convergence iterations (table only)
                conv_iter = max_iters
                for i, val in enumerate(losses):
                    if val < tol_here:
                        conv_iter = i+1
                        break

                final_loss = float(np.min(losses))
                acc_like = safe_accuracy(final_loss)

                stats[opt_name]["loss_history"].append(losses)
                stats[opt_name]["final_losses"].append(final_loss)
                stats[opt_name]["times"].append(duration)
                stats[opt_name]["iters"].append(conv_iter)
                stats[opt_name]["acc"].append(acc_like)

        generate_plots_multi(name, stats)

        print(f"{'Optimizer':12} | {'Loss(mean)':12} | {'Loss(std)':10} | "
              f"{'Iter(mean)':12} | {'Iter(std)':10} | {'Acc(mean)':10} | {'Acc(std)':10} | {'Time(mean)':10}")
        print("-"*118)

        for opt in optimizers:
            ml = np.mean(stats[opt]["final_losses"])
            sl = np.std(stats[opt]["final_losses"])
            mi = np.mean(stats[opt]["iters"])
            si = np.std(stats[opt]["iters"])
            ma = np.mean(stats[opt]["acc"])
            sa = np.std(stats[opt]["acc"])
            mt = np.mean(stats[opt]["times"])

            print(f"{opt:12} | {ml:12.3e} | {sl:10.2e} | {mi:12.1f} | {si:10.1f} | "
                  f"{ma:10.4f} | {sa:10.4f} | {mt:10.3f}")


# =========================
if __name__ == "__main__":
    run_benchmark_suite_multi()


Running MULTI-SEED benchmark...


=== Well-Conditioned Quadratic across 10 seeds ===

Optimizer    | Loss(mean)   | Loss(std)  | Iter(mean)   | Iter(std)  | Acc(mean)  | Acc(std)   | Time(mean)
----------------------------------------------------------------------------------------------------------------------
SGD+Mom      |    6.262e-20 |   8.79e-20 |        568.9 |      100.0 |     1.0000 |     0.0000 |      0.012
Adam         |    3.307e-10 |   9.92e-10 |       1040.9 |      246.0 |     1.0000 |     0.0000 |      0.019
RMSProp      |    4.065e-04 |   3.13e-05 |       2000.0 |        0.0 |     0.9996 |     0.0000 |      0.015
Adam-E       |    3.307e-10 |   9.92e-10 |       1040.9 |      246.0 |     1.0000 |     0.0000 |      0.186

=== Ill-Conditioned Quadratic across 10 seeds ===

Optimizer    | Loss(mean)   | Loss(std)  | Iter(mean)   | Iter(std)  | Acc(mean)  | Acc(std)   | Time(mean)
----------------------------------------------------------------------------------------------

### Short Term Tests

In [ ]:
# --- CONFIGURATION ---
SAVE_PLOTS = True
PLOT_DIR = "./adame_st_results"
if SAVE_PLOTS and not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

# Auto-detect device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Running on Device: {device}")

# ==========================================
# 1. UTILITIES & SEEDING
# ==========================================
def set_seed(seed=67):
    """Enforces strict determinism for fair comparison."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

def flatten_params(param_list): return torch.cat([p.view(-1) for p in param_list])
def unflatten_params(flat, templ):
    restored = []
    idx = 0
    for p in templ:
        numel = p.numel()
        restored.append(flat[idx:idx+numel].view_as(p))
        idx += numel
    return restored
def set_params(model, params):
    for p, new_p in zip(model.parameters(), params): p.data.copy_(new_p)
def get_params(model): return [p.data.clone().detach() for p in model.parameters()]

# ==========================================
# 2. ADAM-E OPTIMIZER
# ==========================================
class AdamE:
    def __init__(self, model, lr=0.001, initial_explorers=6, max_explorers=12):
        self.lr = lr
        self.max_explorers = max_explorers
        self.explorers = []
        
        base = get_params(model)
        for i in range(initial_explorers):
            p = [x.clone() for x in base]
            self.explorers.append({
                "params": p,
                "m": [torch.zeros_like(x) for x in p],
                "v": [torch.zeros_like(x) for x in p],
                "d0": [torch.randn_like(x) for x in p],
                "n": 0,
                "losses": [],
                "is_anchor": (i == 0) # Explorer 0 is the Anchor (Immutable)
            })

    def _perturb(self, grads, scale=0.05):
        flat_g = flatten_params(grads)
        noise = torch.randn_like(flat_g)
        grad_norm = flat_g.norm()
        if grad_norm > 1e-8:
            # Orthogonal projection
            proj = (torch.dot(noise, flat_g) / (grad_norm**2)) * flat_g
            noise = noise - proj
        noise = noise / (noise.norm() + 1e-8) * scale
        return unflatten_params(noise, grads)

    def step(self, model, data, target, criterion):
        batch_losses = []
        grads_buffer = [] 
        
        # --- Update Phase ---
        for e in self.explorers:
            set_params(model, e["params"])
            model.zero_grad()
            out = model(data)
            loss = criterion(out, target)
            loss.backward()
            
            # Gradient Clipping for Stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            e["n"] += 1
            e["losses"].append(loss.item())
            batch_losses.append(loss.item())
            
            grads = [p.grad.detach() for p in model.parameters()]
            if e["is_anchor"]: grads_buffer = grads
            
            for p, g, m, v, d0 in zip(e["params"], grads, e["m"], e["v"], e["d0"]):
                m.mul_(0.9).add_(g, alpha=0.1)
                v.mul_(0.999).addcmul_(g, g, value=0.001)
                m_hat = m / (1 - 0.9**e["n"])
                v_hat = v / (1 - 0.999**e["n"])
                adam_step = -self.lr * m_hat / (v_hat.sqrt() + 1e-8)
                
                if not e["is_anchor"]:
                    adam_step.add_(d0 * self.lr * 0.01) # Constant exploration
                p.add_(adam_step)
        
        # --- Population Dynamics ---
        # 1. Pruning
        if len(self.explorers) > 1:
            losses = batch_losses
            best_idx = np.argmin(losses)
            worst_idx = np.argmax(losses)
            
            if not self.explorers[worst_idx]["is_anchor"]:
                if losses[worst_idx] > losses[best_idx]:
                    best_params = self.explorers[best_idx]["params"]
                    new_params = [x.clone() for x in best_params]
                    if len(grads_buffer) > 0:
                        perturb = self._perturb(grads_buffer, scale=0.05)
                        for p, pert in zip(new_params, perturb): p.add_(pert)
                    self.explorers[worst_idx]["params"] = new_params
                    self.explorers[worst_idx]["m"] = [torch.zeros_like(x) for x in new_params]
                    self.explorers[worst_idx]["v"] = [torch.zeros_like(x) for x in new_params]
                    self.explorers[worst_idx]["n"] = self.explorers[best_idx]["n"]

        # 2. Splitting
        if len(self.explorers) < self.max_explorers and min(batch_losses) < 2.0:
             best_idx = np.argmin(batch_losses)
             best_e = self.explorers[best_idx]
             new_params = [x.clone() for x in best_e["params"]]
             if len(grads_buffer) > 0:
                 perturb = self._perturb(grads_buffer, scale=0.1) 
                 for p, pert in zip(new_params, perturb): p.add_(pert)
             
             self.explorers.append({
                "params": new_params,
                "m": [x.clone() for x in best_e["m"]],
                "v": [x.clone() for x in best_e["v"]],
                "d0": [torch.randn_like(x) for x in new_params],
                "n": best_e["n"],
                "losses": [],
                "is_anchor": False
             })
        return batch_losses

    def evaluate_population(self, model, data_loader, top_k=3):
        """Returns comprehensive stats including Best Scout and Vote Ensemble."""
        results = {}
        scout_accs = []
        all_logits = []
        
        # 1. Evaluate Individual Scouts
        for i, e in enumerate(self.explorers):
            set_params(model, e["params"])
            model.eval()
            correct = 0
            scout_logits = []
            with torch.no_grad():
                for d, t in data_loader:
                    d, t = d.to(device), t.to(device)
                    out = model(d)
                    scout_logits.append(out.cpu())
                    correct += out.argmax(1).eq(t).sum().item()
            
            acc = 100.0 * correct / len(data_loader.dataset)
            scout_accs.append(acc)
            all_logits.append(torch.cat(scout_logits))
            
            if e["is_anchor"]: results["anchor_acc"] = acc

        results["scout_list"] = scout_accs
        results["best_scout_acc"] = max(scout_accs)
        
        # 2. Vote Ensemble (Top K by training loss)
        candidates = [i for i in range(len(self.explorers))]
        candidates.sort(key=lambda i: self.explorers[i]["losses"][-1] if self.explorers[i]["losses"] else float('inf'))
        
        top_indices = candidates[:top_k]
        selected_logits = torch.stack([all_logits[i] for i in top_indices])
        mean_logits = torch.mean(selected_logits, dim=0)
        
        if hasattr(data_loader.dataset, 'targets'): targets = data_loader.dataset.targets
        else: targets = data_loader.dataset.y
        if isinstance(targets, torch.Tensor): targets = targets.cpu()
        else: targets = torch.tensor(targets)
        
        ens_correct = mean_logits.argmax(1).eq(targets).sum().item()
        results["ensemble_acc"] = 100.0 * ens_correct / len(targets)
        
        return results

# ==========================================
# 3. EXPERIMENT RUNNER & VISUALIZER
# ==========================================
def generate_plots(name, adam_loss_hist, adame_loss_hist, metrics, times):
    # Set up a figure with 3 subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # --- Plot 1: Loss Convergence ---
    axes[0].plot(adam_loss_hist, label='Adam', color='red', linestyle='--')
    # Label set to 'Best' because we pass min(losses) from the run loop
    axes[0].plot(adame_loss_hist, label='Adam-E (Best)', color='blue')
    axes[0].set_title(f"{name}: Loss Convergence")
    axes[0].set_xlabel("Epochs")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # --- Plot 2: Final Accuracy (Max Performance) ---
    # We now plot ALL components to show the full picture
    acc_labels = ['Adam', 'Adam-E\nAnchor', 'Adam-E\nBest Scout', 'Adam-E\nEnsemble']
    acc_values = [
        metrics['adam_acc'], 
        metrics['adame_anchor'], 
        metrics['adame_best'], 
        metrics['adame_ens']
    ]
    acc_colors = ['red', 'lightblue', 'blue', 'navy']
    
    bars = axes[1].bar(acc_labels, acc_values, color=acc_colors)
    axes[1].set_title(f"{name}: Accuracy Breakdown")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_ylim(0, 105)
    
    # Add text numbers on bars
    for bar in bars:
        yval = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')

    # --- Plot 3: Time Taken ---
    time_labels = ['Adam', 'Adam-E']
    time_values = [times['adam'], times['adame']]
    time_colors = ['red', 'navy']
    
    time_bars = axes[2].bar(time_labels, time_values, color=time_colors)
    axes[2].set_title(f"{name}: Training Time")
    axes[2].set_ylabel("Time (Seconds)")
    
    # Add text numbers on bars
    for bar in time_bars:
        yval = bar.get_height()
        axes[2].text(bar.get_x() + bar.get_width()/2, yval + 0.1, f"{yval:.1f}s", ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    filename = os.path.join(PLOT_DIR, f"{name.replace(' ', '_')}_results.png")
    plt.savefig(filename)
    plt.close()
    print(f"Graphs saved to: {filename}")

def run_experiment(name, model_class, get_data_func, epochs=5, lr=0.001, seed=67):
    print(f"\n{'='*70}")
    print(f"STARTING EXPERIMENT: {name}")
    print(f"{'='*70}")
    
    # --- A. Run Adam ---
    print("Running Adam...")
    set_seed(seed)
    train_loader, test_loader = get_data_func()
    model = model_class().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    
    adam_losses = []
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for d, t in train_loader:
            d, t = d.to(device), t.to(device)
            opt.zero_grad()
            out = model(d)
            loss = crit(out, t)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            epoch_loss += loss.item()
        adam_losses.append(epoch_loss / len(train_loader))
    
    adam_time = time.time() - start_time
    
    model.eval()
    correct = 0
    with torch.no_grad():
        for d, t in test_loader:
            d, t = d.to(device), t.to(device)
            correct += model(d).argmax(1).eq(t).sum().item()
    adam_acc = 100. * correct / len(test_loader.dataset)

    # --- B. Run Adam-E ---
    print("Running Adam-E...")
    set_seed(seed) # Strict Reset
    train_loader, test_loader = get_data_func() 
    model = model_class().to(device)
    opt_e = AdamE(model, lr=lr)
    
    adame_losses = []
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for d, t in train_loader:
            d, t = d.to(device), t.to(device)
            # The step function returns losses for ALL explorers in the batch
            losses = opt_e.step(model, d, t, crit)
            # We record the MINIMUM (Best) loss for the graph
            epoch_loss += min(losses)
        adame_losses.append(epoch_loss / len(train_loader))
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"  [Epoch {epoch+1}/{epochs}] Best Loss: {adame_losses[-1]:.4f}")
            
    adame_time = time.time() - start_time
    
    # Evaluate
    stats = opt_e.evaluate_population(model, test_loader, top_k=3)
    
    # --- C. Reporting ---
    print("-" * 40)
    print(f"Time | Adam: {adam_time:.1f}s | Adam-E: {adame_time:.1f}s")
    print("-" * 40)
    print(f"Adam Accuracy:         {adam_acc:.2f}%")
    print(f"Adam-E Anchor:         {stats['anchor_acc']:.2f}%")
    print(f"Adam-E Best Scout:     {stats['best_scout_acc']:.2f}%")
    print(f"Adam-E Ensemble:       {stats['ensemble_acc']:.2f}%")
    print("-" * 40)
    print(f"Diversity List: {['{:.2f}'.format(x) for x in stats['scout_list']]}")

    # Generate Plots
    metrics = {
        'adam_acc': adam_acc,
        'adame_anchor': stats['anchor_acc'],
        'adame_best': stats['best_scout_acc'],
        'adame_ens': stats['ensemble_acc']
    }
    
    times = {
        'adam': adam_time,
        'adame': adame_time
    }
    
    generate_plots(name, adam_losses, adame_losses, metrics, times)

# ==========================================
# 4. DATASETS & MODELS
# ==========================================
# --- Helpers ---
class CorruptedMNIST(datasets.MNIST):
    def __init__(self, *args, corruption_prob=0.4, **kwargs):
        super().__init__(*args, **kwargs)
        torch.manual_seed(67) # Consistent Corruption
        labels = self.targets.clone()
        mask = torch.rand(len(labels)) < corruption_prob
        random_labels = torch.randint(0, 10, (len(labels),))
        self.targets[mask] = random_labels[mask]

set_seed(67); perm = torch.randperm(784) 
class PermutedMNIST(datasets.MNIST):
    def __getitem__(self, index):
        img, target = super().__getitem__(index)
        img = img.view(-1)[perm].view(784, 1)
        return img, target

# --- Getters ---
def get_parity():
    N = 2000
    X = torch.randint(0, 2, (N, 12)).float()
    y = X.sum(dim=1).long() % 2
    dataset = TensorDataset(X, y)
    dataset.y = y
    return DataLoader(dataset, batch_size=64, shuffle=True), DataLoader(dataset, batch_size=2000)

def get_spirals():
    N = 2000
    theta = np.sqrt(np.random.rand(N)) * 6 * np.pi 
    r_a, r_b = theta, -theta
    data_a = np.array([np.cos(theta)*r_a, np.sin(theta)*r_a]).T + np.random.randn(N,2)*0.4
    data_b = np.array([np.cos(theta)*r_b, np.sin(theta)*r_b]).T + np.random.randn(N,2)*0.4
    X = np.concatenate([data_a, data_b])
    y = np.concatenate([np.zeros(N), np.ones(N)])
    X = (X - X.mean(axis=0)) / X.std(axis=0)
    tensor_x, tensor_y = torch.FloatTensor(X), torch.LongTensor(y)
    dataset = TensorDataset(tensor_x, tensor_y)
    dataset.y = tensor_y 
    return DataLoader(dataset, batch_size=64, shuffle=True), DataLoader(dataset, batch_size=2000)

def get_permuted_mnist():
    t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    tr = PermutedMNIST('./data', train=True, download=True, transform=t)
    te = PermutedMNIST('./data', train=False, download=True, transform=t)
    return DataLoader(tr, batch_size=64, shuffle=True), DataLoader(te, batch_size=1000)

def get_corrupted_mnist():
    t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.13,), (0.3,))])
    tr = CorruptedMNIST('./data', train=True, download=True, transform=t, corruption_prob=0.4)
    te = datasets.MNIST('./data', train=False, download=True, transform=t)
    return DataLoader(tr, batch_size=128, shuffle=True), DataLoader(te, batch_size=1000)

def get_fashion():
    t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    tr = datasets.FashionMNIST('./data', train=True, download=True, transform=t)
    te = datasets.FashionMNIST('./data', train=False, download=True, transform=t)
    return DataLoader(tr, batch_size=128, shuffle=True), DataLoader(te, batch_size=1000)

def get_mnist():
    t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.13,), (0.3,))])
    tr = datasets.MNIST('./data', train=True, download=True, transform=t)
    te = datasets.MNIST('./data', train=False, download=True, transform=t)
    return DataLoader(tr, batch_size=128, shuffle=True), DataLoader(te, batch_size=1000)

def get_cifar():
    t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    tr = datasets.CIFAR10('./data', train=True, download=True, transform=t)
    te = datasets.CIFAR10('./data', train=False, download=True, transform=t)
    return DataLoader(tr, batch_size=128, shuffle=True), DataLoader(te, batch_size=1000)

# --- Models ---
class ParityNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(12, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))
    def forward(self, x): return self.net(x)

class SimpleRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(1, 128, batch_first=True)
        self.fc = nn.Linear(128, 10)
    def forward(self, x):
        _, hn = self.rnn(x); return self.fc(hn.squeeze(0))

class StarvedSpiralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 8), nn.Tanh(), nn.Linear(8, 2))
    def forward(self, x): return self.net(x)

class StandardSpiralNet(nn.Module): # Big Capacity
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
    def forward(self, x): return self.net(x)

class StarvedNet(nn.Module): # Bottleneck
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(784, 30), nn.ReLU(), nn.Linear(30, 10))
    def forward(self, x): return self.net(x)

class SimpleCNN(nn.Module): # Standard Benchmark
    def __init__(self, in_channels=1): 
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64*5*5 if in_channels==1 else 64*6*6, 128), nn.ReLU(), nn.Linear(128, 10)
        )
    def forward(self, x): return self.net(x)

# ==========================================
# 5. MAIN EXECUTION (UNCOMMENT TO RUN)
# ==========================================
if __name__ == "__main__":
    
    # 1. PARITY (Rugged)
    # run_experiment("12-bit Parity", ParityNet, get_parity, epochs=30, lr=0.01)
    
    # 2. PERMUTED MNIST (Torture Test - Vanishing Gradients)
    # run_experiment("Permuted MNIST (RNN)", SimpleRNN, get_permuted_mnist, epochs=3, lr=0.001)

    # 3. STARVED SPIRALS (Visualizing Local Optima)
    # run_experiment("Starved Spirals", StarvedSpiralNet, get_spirals, epochs=30, lr=0.02)

    # 4. HARD SPIRALS (Big Network Speed Test)
    # run_experiment("Pure Hard Spirals (Big Net)", StandardSpiralNet, get_spirals, epochs=30, lr=0.02)

    # 5. STARVED FASHION MNIST (Bottleneck)
    # run_experiment("Starved FashionMNIST", StarvedNet, get_fashion, epochs=5, lr=0.002)

    # 6. STARVED MNIST (Bottleneck)
    # run_experiment("Starved MNIST", StarvedNet, get_mnist, epochs=5, lr=0.002)

    # 7. STANDARD FASHION MNIST (CNN Benchmark)
    # run_experiment("Standard FashionMNIST", lambda: SimpleCNN(in_channels=1), get_fashion, epochs=5, lr=0.001)
    
    # 8. STANDARD MNIST (CNN Benchmark)
    # run_experiment("Standard MNIST", lambda: SimpleCNN(in_channels=1), get_mnist, epochs=3, lr=0.001)

    # 9. CIFAR-10 (Standard Benchmark)
    # run_experiment("CIFAR-10", lambda: SimpleCNN(in_channels=3), get_cifar, epochs=10, lr=0.001)
    
    # 10. CORRUPTED MNIST (Noise Robustness Test)
    # run_experiment("MNIST (40% Label Corruption)", lambda: SimpleCNN(in_channels=1), get_corrupted_mnist, epochs=10, lr=0.001)

Running on Device: cuda

STARTING EXPERIMENT: 12-bit Parity
Running Adam...
Running Adam-E...
  [Epoch 1/30] Best Loss: 0.6912
  [Epoch 5/30] Best Loss: 0.6733
  [Epoch 10/30] Best Loss: 0.5832
  [Epoch 15/30] Best Loss: 0.3242
  [Epoch 20/30] Best Loss: 0.2170
  [Epoch 25/30] Best Loss: 0.1562
  [Epoch 30/30] Best Loss: 0.1155
----------------------------------------
Time | Adam: 1.1s | Adam-E: 13.8s
----------------------------------------
Adam Accuracy:         93.45%
Adam-E Anchor:         93.10%
Adam-E Best Scout:     93.40%
Adam-E Ensemble:       97.10%
----------------------------------------
Diversity List: ['93.10', '73.15', '90.35', '90.95', '88.10', '81.70', '93.40', '81.85', '87.00', '90.65', '89.75', '93.40']
Graphs saved to: ./adame_st_results/12-bit_Parity_results.png

STARTING EXPERIMENT: Permuted MNIST (RNN)
Running Adam...
Running Adam-E...
  [Epoch 1/3] Best Loss: 1.4002
----------------------------------------
Time | Adam: 29.3s | Adam-E: 172.1s
--------------------

### Long Term Tests

In [ ]:
# --- CONFIGURATION ---
SAVE_PLOTS = True
PLOT_DIR = "./adame_lt_results"
if SAVE_PLOTS and not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

# Auto-detect device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Running on Device: {device}")

# ==========================================
# 1. UTILITIES & SEEDING
# ==========================================
def set_seed(seed=67):
    """Enforces strict determinism for fair comparison."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

def flatten_params(param_list): return torch.cat([p.view(-1) for p in param_list])
def unflatten_params(flat, templ):
    restored = []
    idx = 0
    for p in templ:
        numel = p.numel()
        restored.append(flat[idx:idx+numel].view_as(p))
        idx += numel
    return restored
def set_params(model, params):
    for p, new_p in zip(model.parameters(), params): p.data.copy_(new_p)
def get_params(model): return [p.data.clone().detach() for p in model.parameters()]

# ==========================================
# 2. ADAM-E OPTIMIZER
# ==========================================
class AdamE:
    def __init__(self, model, lr=0.001, initial_explorers=6, max_explorers=12):
        self.lr = lr
        self.max_explorers = max_explorers
        self.explorers = []
        
        base = get_params(model)
        for i in range(initial_explorers):
            p = [x.clone() for x in base]
            self.explorers.append({
                "params": p,
                "m": [torch.zeros_like(x) for x in p],
                "v": [torch.zeros_like(x) for x in p],
                "d0": [torch.randn_like(x) for x in p],
                "n": 0,
                "losses": [],
                "is_anchor": (i == 0) # Explorer 0 is the Anchor (Immutable)
            })

    def _perturb(self, grads, scale=0.05):
        flat_g = flatten_params(grads)
        noise = torch.randn_like(flat_g)
        grad_norm = flat_g.norm()
        if grad_norm > 1e-8:
            # Orthogonal projection
            proj = (torch.dot(noise, flat_g) / (grad_norm**2)) * flat_g
            noise = noise - proj
        noise = noise / (noise.norm() + 1e-8) * scale
        return unflatten_params(noise, grads)

    def step(self, model, data, target, criterion):
        batch_losses = []
        grads_buffer = [] 
        
        # --- Update Phase ---
        for e in self.explorers:
            set_params(model, e["params"])
            model.zero_grad()
            out = model(data)
            loss = criterion(out, target)
            loss.backward()
            
            # Gradient Clipping for Stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            e["n"] += 1
            e["losses"].append(loss.item())
            batch_losses.append(loss.item())
            
            grads = [p.grad.detach() for p in model.parameters()]
            if e["is_anchor"]: grads_buffer = grads
            
            for p, g, m, v, d0 in zip(e["params"], grads, e["m"], e["v"], e["d0"]):
                m.mul_(0.9).add_(g, alpha=0.1)
                v.mul_(0.999).addcmul_(g, g, value=0.001)
                m_hat = m / (1 - 0.9**e["n"])
                v_hat = v / (1 - 0.999**e["n"])
                adam_step = -self.lr * m_hat / (v_hat.sqrt() + 1e-8)
                
                if not e["is_anchor"]:
                    adam_step.add_(d0 * self.lr * 0.01) # Constant exploration
                p.add_(adam_step)
        
        # --- Population Dynamics ---
        # 1. Pruning
        if len(self.explorers) > 1:
            losses = batch_losses
            best_idx = np.argmin(losses)
            worst_idx = np.argmax(losses)
            
            if not self.explorers[worst_idx]["is_anchor"]:
                if losses[worst_idx] > losses[best_idx]:
                    best_params = self.explorers[best_idx]["params"]
                    new_params = [x.clone() for x in best_params]
                    if len(grads_buffer) > 0:
                        perturb = self._perturb(grads_buffer, scale=0.05)
                        for p, pert in zip(new_params, perturb): p.add_(pert)
                    self.explorers[worst_idx]["params"] = new_params
                    self.explorers[worst_idx]["m"] = [torch.zeros_like(x) for x in new_params]
                    self.explorers[worst_idx]["v"] = [torch.zeros_like(x) for x in new_params]
                    self.explorers[worst_idx]["n"] = self.explorers[best_idx]["n"]

        # 2. Splitting
        if len(self.explorers) < self.max_explorers and min(batch_losses) < 2.0:
             best_idx = np.argmin(batch_losses)
             best_e = self.explorers[best_idx]
             new_params = [x.clone() for x in best_e["params"]]
             if len(grads_buffer) > 0:
                 perturb = self._perturb(grads_buffer, scale=0.1) 
                 for p, pert in zip(new_params, perturb): p.add_(pert)
             
             self.explorers.append({
                "params": new_params,
                "m": [x.clone() for x in best_e["m"]],
                "v": [x.clone() for x in best_e["v"]],
                "d0": [torch.randn_like(x) for x in new_params],
                "n": best_e["n"],
                "losses": [],
                "is_anchor": False
             })
        return batch_losses

    def evaluate_population(self, model, data_loader, top_k=3):
        """Returns comprehensive stats including Best Scout and Vote Ensemble."""
        results = {}
        scout_accs = []
        all_logits = []
        
        # 1. Evaluate Individual Scouts
        for i, e in enumerate(self.explorers):
            set_params(model, e["params"])
            model.eval()
            correct = 0
            scout_logits = []
            with torch.no_grad():
                for d, t in data_loader:
                    d, t = d.to(device), t.to(device)
                    out = model(d)
                    scout_logits.append(out.cpu())
                    # FIXED: Comparison stays on device
                    correct += out.argmax(1).eq(t).sum().item()
            
            acc = 100.0 * correct / len(data_loader.dataset)
            scout_accs.append(acc)
            all_logits.append(torch.cat(scout_logits))
            
            if e["is_anchor"]: results["anchor_acc"] = acc

        results["scout_list"] = scout_accs
        results["best_scout_acc"] = max(scout_accs)
        
        # 2. Vote Ensemble (Top K by training loss)
        candidates = [i for i in range(len(self.explorers))]
        # Sort by most recent training loss (proxy for quality)
        candidates.sort(key=lambda i: self.explorers[i]["losses"][-1] if self.explorers[i]["losses"] else float('inf'))
        
        top_indices = candidates[:top_k]
        selected_logits = torch.stack([all_logits[i] for i in top_indices])
        mean_logits = torch.mean(selected_logits, dim=0)
        
        if hasattr(data_loader.dataset, 'targets'): targets = data_loader.dataset.targets
        else: targets = data_loader.dataset.y
        if isinstance(targets, torch.Tensor): targets = targets.cpu()
        else: targets = torch.tensor(targets)
        
        ens_correct = mean_logits.argmax(1).eq(targets).sum().item()
        results["ensemble_acc"] = 100.0 * ens_correct / len(targets)
        
        return results

# ==========================================
# 3. EXPERIMENT RUNNER & VISUALIZER
# ==========================================
def generate_plots(name, adam_loss_hist, adame_loss_hist, metrics, times):
    # Set up a figure with 3 subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # --- Plot 1: Loss Convergence ---
    axes[0].plot(adam_loss_hist, label='Adam', color='red', linestyle='--')
    # Label set to 'Best' because we pass min(losses) from the run loop
    axes[0].plot(adame_loss_hist, label='Adam-E (Best)', color='blue')
    axes[0].set_title(f"{name}: Loss Convergence")
    axes[0].set_xlabel("Epochs")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # --- Plot 2: Final Accuracy (Max Performance) ---
    # We now plot ALL components to show the full picture
    acc_labels = ['Adam', 'Adam-E\nAnchor', 'Adam-E\nBest Scout', 'Adam-E\nEnsemble']
    acc_values = [
        metrics['adam_acc'], 
        metrics['adame_anchor'], 
        metrics['adame_best'], 
        metrics['adame_ens']
    ]
    acc_colors = ['red', 'lightblue', 'blue', 'navy']
    
    bars = axes[1].bar(acc_labels, acc_values, color=acc_colors)
    axes[1].set_title(f"{name}: Accuracy Breakdown")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_ylim(0, 105)
    
    # Add text numbers on bars
    for bar in bars:
        yval = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')

    # --- Plot 3: Time Taken ---
    time_labels = ['Adam', 'Adam-E']
    time_values = [times['adam'], times['adame']]
    time_colors = ['red', 'navy']
    
    time_bars = axes[2].bar(time_labels, time_values, color=time_colors)
    axes[2].set_title(f"{name}: Training Time")
    axes[2].set_ylabel("Time (Seconds)")
    
    # Add text numbers on bars
    for bar in time_bars:
        yval = bar.get_height()
        axes[2].text(bar.get_x() + bar.get_width()/2, yval + 0.1, f"{yval:.1f}s", ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    filename = os.path.join(PLOT_DIR, f"{name.replace(' ', '_')}_results.png")
    plt.savefig(filename)
    plt.close()
    print(f"Graphs saved to: {filename}")

def run_experiment(name, model_class, get_data_func, epochs=5, lr=0.001, seed=67):
    print(f"\n{'='*70}")
    print(f"STARTING EXPERIMENT: {name}")
    print(f"{'='*70}")
    
    # --- A. Run Adam ---
    print("Running Adam...")
    set_seed(seed)
    train_loader, test_loader = get_data_func()
    model = model_class().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    
    adam_losses = []
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for d, t in train_loader:
            d, t = d.to(device), t.to(device)
            opt.zero_grad()
            out = model(d)
            loss = crit(out, t)
            loss.backward()
            # Clipping for Adam to ensure fair comparison
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            epoch_loss += loss.item()
        adam_losses.append(epoch_loss / len(train_loader))
    
    adam_time = time.time() - start_time
    
    model.eval()
    correct = 0
    with torch.no_grad():
        for d, t in test_loader:
            d, t = d.to(device), t.to(device)
            correct += model(d).argmax(1).eq(t).sum().item()
    adam_acc = 100. * correct / len(test_loader.dataset)

    # --- B. Run Adam-E ---
    print("Running Adam-E...")
    set_seed(seed) # Strict Reset
    train_loader, test_loader = get_data_func() 
    model = model_class().to(device)
    opt_e = AdamE(model, lr=lr)
    
    adame_losses = []
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for d, t in train_loader:
            d, t = d.to(device), t.to(device)
            # The step function returns losses for ALL explorers in the batch
            losses = opt_e.step(model, d, t, crit)
            # We record the MINIMUM (Best) loss for the graph
            epoch_loss += min(losses)
        adame_losses.append(epoch_loss / len(train_loader))
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"  [Epoch {epoch+1}/{epochs}] Best Loss: {adame_losses[-1]:.4f}")
            
    adame_time = time.time() - start_time
    
    # Evaluate
    stats = opt_e.evaluate_population(model, test_loader, top_k=3)
    
    # --- C. Reporting ---
    print("-" * 40)
    print(f"Time | Adam: {adam_time:.1f}s | Adam-E: {adame_time:.1f}s")
    print("-" * 40)
    print(f"Adam Accuracy:         {adam_acc:.2f}%")
    print(f"Adam-E Anchor:         {stats['anchor_acc']:.2f}%")
    print(f"Adam-E Best Scout:     {stats['best_scout_acc']:.2f}%")
    print(f"Adam-E Ensemble:       {stats['ensemble_acc']:.2f}%")
    print("-" * 40)
    print(f"Diversity List: {['{:.2f}'.format(x) for x in stats['scout_list']]}")

    # Generate Plots
    metrics = {
        'adam_acc': adam_acc,
        'adame_anchor': stats['anchor_acc'],
        'adame_best': stats['best_scout_acc'],
        'adame_ens': stats['ensemble_acc']
    }
    
    times = {
        'adam': adam_time,
        'adame': adame_time
    }
    
    generate_plots(name, adam_losses, adame_losses, metrics, times)

# ==========================================
# 4. DATASETS & MODELS
# ==========================================
# --- Helpers ---
class CorruptedMNIST(datasets.MNIST):
    def __init__(self, *args, corruption_prob=0.4, **kwargs):
        super().__init__(*args, **kwargs)
        torch.manual_seed(67) # Consistent Corruption
        labels = self.targets.clone()
        mask = torch.rand(len(labels)) < corruption_prob
        random_labels = torch.randint(0, 10, (len(labels),))
        self.targets[mask] = random_labels[mask]

set_seed(67); perm = torch.randperm(784) 
class PermutedMNIST(datasets.MNIST):
    def __getitem__(self, index):
        img, target = super().__getitem__(index)
        img = img.view(-1)[perm].view(784, 1)
        return img, target

# --- Getters ---
def get_parity():
    N = 2000
    X = torch.randint(0, 2, (N, 12)).float()
    y = X.sum(dim=1).long() % 2
    dataset = TensorDataset(X, y)
    dataset.y = y
    return DataLoader(dataset, batch_size=64, shuffle=True), DataLoader(dataset, batch_size=2000)

def get_spirals():
    N = 2000
    theta = np.sqrt(np.random.rand(N)) * 6 * np.pi 
    r_a, r_b = theta, -theta
    data_a = np.array([np.cos(theta)*r_a, np.sin(theta)*r_a]).T + np.random.randn(N,2)*0.4
    data_b = np.array([np.cos(theta)*r_b, np.sin(theta)*r_b]).T + np.random.randn(N,2)*0.4
    X = np.concatenate([data_a, data_b])
    y = np.concatenate([np.zeros(N), np.ones(N)])
    X = (X - X.mean(axis=0)) / X.std(axis=0)
    tensor_x, tensor_y = torch.FloatTensor(X), torch.LongTensor(y)
    dataset = TensorDataset(tensor_x, tensor_y)
    dataset.y = tensor_y 
    return DataLoader(dataset, batch_size=64, shuffle=True), DataLoader(dataset, batch_size=2000)

def get_permuted_mnist():
    t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    tr = PermutedMNIST('./data', train=True, download=True, transform=t)
    te = PermutedMNIST('./data', train=False, download=True, transform=t)
    return DataLoader(tr, batch_size=64, shuffle=True), DataLoader(te, batch_size=1000)

def get_corrupted_mnist():
    t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.13,), (0.3,))])
    tr = CorruptedMNIST('./data', train=True, download=True, transform=t, corruption_prob=0.4)
    te = datasets.MNIST('./data', train=False, download=True, transform=t)
    return DataLoader(tr, batch_size=128, shuffle=True), DataLoader(te, batch_size=1000)

def get_fashion():
    t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    tr = datasets.FashionMNIST('./data', train=True, download=True, transform=t)
    te = datasets.FashionMNIST('./data', train=False, download=True, transform=t)
    return DataLoader(tr, batch_size=128, shuffle=True), DataLoader(te, batch_size=1000)

def get_mnist():
    t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.13,), (0.3,))])
    tr = datasets.MNIST('./data', train=True, download=True, transform=t)
    te = datasets.MNIST('./data', train=False, download=True, transform=t)
    return DataLoader(tr, batch_size=128, shuffle=True), DataLoader(te, batch_size=1000)

def get_cifar():
    t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    tr = datasets.CIFAR10('./data', train=True, download=True, transform=t)
    te = datasets.CIFAR10('./data', train=False, download=True, transform=t)
    return DataLoader(tr, batch_size=128, shuffle=True), DataLoader(te, batch_size=1000)

# --- Models ---
class ParityNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(12, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))
    def forward(self, x): return self.net(x)

class SimpleRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(1, 128, batch_first=True)
        self.fc = nn.Linear(128, 10)
    def forward(self, x):
        _, hn = self.rnn(x); return self.fc(hn.squeeze(0))

class StarvedSpiralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 8), nn.Tanh(), nn.Linear(8, 2))
    def forward(self, x): return self.net(x)

class StandardSpiralNet(nn.Module): # Big Capacity
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
    def forward(self, x): return self.net(x)

class StarvedNet(nn.Module): # Bottleneck
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(784, 30), nn.ReLU(), nn.Linear(30, 10))
    def forward(self, x): return self.net(x)

class SimpleCNN(nn.Module): # Standard Benchmark
    def __init__(self, in_channels=1): 
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64*5*5 if in_channels==1 else 64*6*6, 128), nn.ReLU(), nn.Linear(128, 10)
        )
    def forward(self, x): return self.net(x)

# ==========================================
# 5. MAIN EXECUTION (UNCOMMENT TO RUN)
# ==========================================
if __name__ == "__main__":
    
    # 1. PARITY (Rugged)
    # run_experiment("12-bit Parity", ParityNet, get_parity, epochs=100, lr=0.01)
    
    # 2. PERMUTED MNIST (Torture Test - Vanishing Gradients)
    # run_experiment("Permuted MNIST (RNN)", SimpleRNN, get_permuted_mnist, epochs=15, lr=0.001)

    # 3. STARVED SPIRALS (Visualizing Local Optima)
    # run_experiment("Starved Spirals", StarvedSpiralNet, get_spirals, epochs=100, lr=0.02)

    # 4. HARD SPIRALS (Big Network Speed Test)
    # run_experiment("Pure Hard Spirals (Big Net)", StandardSpiralNet, get_spirals, epochs=100, lr=0.02)

    # 5. STARVED FASHION MNIST (Bottleneck)
    # run_experiment("Starved FashionMNIST", StarvedNet, get_fashion, epochs=10, lr=0.002)

    # 6. STARVED MNIST (Bottleneck)
    # run_experiment("Starved MNIST", StarvedNet, get_mnist, epochs=10, lr=0.002)

    # 7. STANDARD FASHION MNIST (CNN Benchmark)
    # run_experiment("Standard FashionMNIST", lambda: SimpleCNN(in_channels=1), get_fashion, epochs=10, lr=0.001)
    
    # 8. STANDARD MNIST (CNN Benchmark)
    # run_experiment("Standard MNIST", lambda: SimpleCNN(in_channels=1), get_mnist, epochs=20, lr=0.001)

    # 9. CIFAR-10 (Standard Benchmark)
    # run_experiment("CIFAR-10", lambda: SimpleCNN(in_channels=3), get_cifar, epochs=20, lr=0.001)
    
    # 10. CORRUPTED MNIST (Noise Robustness Test)
    # run_experiment("MNIST (40% Label Corruption)", lambda: SimpleCNN(in_channels=1), get_corrupted_mnist, epochs=20, lr=0.001)

Running on Device: cuda

STARTING EXPERIMENT: 12-bit Parity
Running Adam...
Running Adam-E...
  [Epoch 1/100] Best Loss: 0.6912
  [Epoch 5/100] Best Loss: 0.6733
  [Epoch 10/100] Best Loss: 0.5832
  [Epoch 15/100] Best Loss: 0.3242
  [Epoch 20/100] Best Loss: 0.2170
  [Epoch 25/100] Best Loss: 0.1562
  [Epoch 30/100] Best Loss: 0.1155
  [Epoch 35/100] Best Loss: 0.0897
  [Epoch 40/100] Best Loss: 0.0725
  [Epoch 45/100] Best Loss: 0.0510
  [Epoch 50/100] Best Loss: 0.0399
  [Epoch 55/100] Best Loss: 0.0296
  [Epoch 60/100] Best Loss: 0.0308
  [Epoch 65/100] Best Loss: 0.0213
  [Epoch 70/100] Best Loss: 0.0161
  [Epoch 75/100] Best Loss: 0.0151
  [Epoch 80/100] Best Loss: 0.0134
  [Epoch 85/100] Best Loss: 0.0086
  [Epoch 90/100] Best Loss: 0.0067
  [Epoch 95/100] Best Loss: 0.0066
  [Epoch 100/100] Best Loss: 0.0106
----------------------------------------
Time | Adam: 3.6s | Adam-E: 43.4s
----------------------------------------
Adam Accuracy:         99.65%
Adam-E Anchor:         99.

100%|██████████| 9.91M/9.91M [00:06<00:00, 1.49MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 103kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 840kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.6MB/s]


Running Adam-E...
  [Epoch 1/15] Best Loss: 1.4002
  [Epoch 5/15] Best Loss: 0.7510
  [Epoch 10/15] Best Loss: 0.7583
  [Epoch 15/15] Best Loss: 1.0295
----------------------------------------
Time | Adam: 104.2s | Adam-E: 482.6s
----------------------------------------
Adam Accuracy:         73.05%
Adam-E Anchor:         73.90%
Adam-E Best Scout:     73.90%
Adam-E Ensemble:       68.07%
----------------------------------------
Diversity List: ['73.90', '34.72', '45.26', '32.88', '50.35', '50.65', '49.06', '45.71', '73.87', '31.58', '35.45', '52.32']
Graphs saved to: ./adame_lt_results/Permuted_MNIST_(RNN)_results.png

STARTING EXPERIMENT: Starved Spirals
Running Adam...
Running Adam-E...
  [Epoch 1/100] Best Loss: 0.6660
  [Epoch 5/100] Best Loss: 0.6243
  [Epoch 10/100] Best Loss: 0.4814
  [Epoch 15/100] Best Loss: 0.4166
  [Epoch 20/100] Best Loss: 0.3872
  [Epoch 25/100] Best Loss: 0.3422
  [Epoch 30/100] Best Loss: 0.2710
  [Epoch 35/100] Best Loss: 0.2455
  [Epoch 40/100] Best Lo

100%|██████████| 26.4M/26.4M [00:05<00:00, 4.82MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 205kB/s]
100%|██████████| 4.42M/4.42M [00:02<00:00, 1.73MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 31.1MB/s]


Running Adam-E...
  [Epoch 1/10] Best Loss: 0.5306
  [Epoch 5/10] Best Loss: 0.3163
  [Epoch 10/10] Best Loss: 0.2742
----------------------------------------
Time | Adam: 44.0s | Adam-E: 92.5s
----------------------------------------
Adam Accuracy:         86.91%
Adam-E Anchor:         86.66%
Adam-E Best Scout:     86.70%
Adam-E Ensemble:       86.77%
----------------------------------------
Diversity List: ['86.66', '86.39', '86.38', '75.63', '86.33', '79.60', '86.19', '86.66', '86.23', '78.09', '86.70', '81.85']
Graphs saved to: ./adame_lt_results/Starved_FashionMNIST_results.png

STARTING EXPERIMENT: Starved MNIST
Running Adam...


...

Running Adam-E...
  [Epoch 1/10] Best Loss: 0.3194
  [Epoch 5/10] Best Loss: 0.0861
  [Epoch 10/10] Best Loss: 0.0511
----------------------------------------
Time | Adam: 44.3s | Adam-E: 93.9s
----------------------------------------
Adam Accuracy:         96.51%
Adam-E Anchor:         96.51%
Adam-E Best Scout:     96.86%
Adam-E Ensemble:       97.19%
----------------------------------------
Diversity List: ['96.51', '93.54', '93.85', '96.76', '94.55', '95.69', '94.62', '93.61', '96.57', '96.80', '96.80', '96.86']
Graphs saved to: ./adame_lt_results/Starved_MNIST_results.png

STARTING EXPERIMENT: Standard FashionMNIST
Running Adam...
Running Adam-E...
  [Epoch 1/10] Best Loss: 0.4736
  [Epoch 5/10] Best Loss: 0.1645
  [Epoch 10/10] Best Loss: 0.0857
----------------------------------------
Time | Adam: 49.9s | Adam-E: 154.0s
----------------------------------------
Adam Accuracy:         91.30%
Adam-E Anchor:         90.73%
Adam-E Best Scout:     91.57%
Adam-E Ensemble:       91.80%
-

100%|██████████| 170M/170M [00:34<00:00, 4.93MB/s] 


Running Adam-E...
  [Epoch 1/20] Best Loss: 1.4082
  [Epoch 5/20] Best Loss: 0.5952
  [Epoch 10/20] Best Loss: 0.2438
  [Epoch 15/20] Best Loss: 0.0784
  [Epoch 20/20] Best Loss: 0.0330
----------------------------------------
Time | Adam: 100.7s | Adam-E: 305.6s
----------------------------------------
Adam Accuracy:         69.79%
Adam-E Anchor:         70.69%
Adam-E Best Scout:     70.69%
Adam-E Ensemble:       71.09%
----------------------------------------
Diversity List: ['70.69', '63.79', '65.82', '69.86', '69.50', '69.57', '63.51', '69.69', '70.33', '64.04', '65.84', '69.49']
Graphs saved to: ./adame_lt_results/CIFAR-10_results.png

STARTING EXPERIMENT: MNIST (40% Label Corruption)
Running Adam...


100%|██████████| 9.91M/9.91M [00:11<00:00, 873kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 107kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 585kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.57MB/s]


Running Adam-E...
  [Epoch 1/20] Best Loss: 1.5674
  [Epoch 5/20] Best Loss: 1.4352
  [Epoch 10/20] Best Loss: 1.2813
  [Epoch 15/20] Best Loss: 1.0485
  [Epoch 20/20] Best Loss: 0.8234
----------------------------------------
Time | Adam: 94.2s | Adam-E: 322.4s
----------------------------------------
Adam Accuracy:         80.75%
Adam-E Anchor:         80.47%
Adam-E Best Scout:     81.47%
Adam-E Ensemble:       89.06%
----------------------------------------
Diversity List: ['80.47', '76.85', '76.53', '76.52', '74.91', '70.89', '80.46', '79.97', '76.33', '73.92', '81.47', '75.29']
Graphs saved to: ./adame_lt_results/MNIST_(40%_Label_Corruption)_results.png
